# Train an offline voice for the Abhyashify assistant

This notebook turns the zip you got from the recorder page into two files, `<name>.onnx` and `<name>.onnx.json`, that the app can run offline.
It fine-tunes a Piper voice, starting from a ready-made voice so a small recording is enough.

**Before you start**
- Use a GPU: *Runtime → Change runtime type → T4 GPU*.
- Only train on a voice whose owner agreed. The zip must contain `consent/consent.wav` and `consent/consent.json`. The notebook stops without them.
- Do one language at a time. Run the notebook again for the other language, and again for the other assistant voice.

> I wrote this from Piper's own training guide (OHF-Voice/piper1-gpl, docs/TRAINING.md) but could not run it, because it needs a GPU. If a step fails, the error message and that guide are the place to look. Listen to the test sentences before you use the voice.

In [ ]:
#@title 1. Settings
LANG = "en"            #@param ["en", "hi"]
VOICE_NAME = "anu_en"  #@param {type:"string"}
EPOCHS = 400           #@param {type:"integer"}
BATCH_SIZE = 16        #@param {type:"integer"}
# espeak voice for the language ("en-us" or "hi"), and which ready-made voice to start from
ESPEAK = {"en": "en-us", "hi": "hi"}[LANG]
BASE_PREFIX = {"en": "en/en_US/lessac/medium/", "hi": "hi/hi_IN/rohan/"}[LANG]   # change this to start from another voice

In [ ]:
#@title 2. Install Piper training (about 5 to 10 minutes)
!apt-get -qq install -y build-essential cmake ninja-build > /dev/null
!git clone -q https://github.com/OHF-voice/piper1-gpl.git
%cd piper1-gpl
!pip -q install -e '.[train]'
!./build_monotonic_align.sh
!python3 setup.py build_ext --inplace
%cd ..

In [ ]:
#@title 3. Upload the zip from the recorder and check it
import zipfile, os, wave, glob
from google.colab import files
up = files.upload()
zname = list(up.keys())[0]
os.makedirs("data", exist_ok=True)
zipfile.ZipFile(zname).extractall("data")
assert os.path.exists("data/consent/consent.wav") and os.path.exists("data/consent/consent.json"), "No consent record in this zip. Stop here: do not train on a voice without it."
import json; print("Consent:", json.load(open("data/consent/consent.json"))["statement"])
rows = [l.rstrip("\n").split("|", 1) for l in open("data/metadata.csv", encoding="utf-8") if l.strip()]
total = 0; bad = []
for f, t in rows:
    p = os.path.join("data/wavs", f)
    try:
        w = wave.open(p); assert w.getframerate() == 22050 and w.getnchannels() == 1; total += w.getnframes() / 22050
    except Exception as e: bad.append(f)
print(len(rows), "sentences,", round(total/60, 1), "minutes of speech;", len(bad), "unusable files", bad[:5])
assert not bad, "Fix or remove the unusable files first."
if total < 10*60: print("WARNING: under 10 minutes. It may work, but more recordings give a better voice.")

In [ ]:
#@title 4. Get the ready-made voice to start from
from huggingface_hub import list_repo_files, hf_hub_download
allf = [f for f in list_repo_files("rhasspy/piper-checkpoints", repo_type="dataset") if f.startswith(BASE_PREFIX) and f.endswith(".ckpt")]
print("\n".join(allf) or "Nothing under " + BASE_PREFIX + ". Look at the dataset page and change BASE_PREFIX.")
BASE_CKPT = hf_hub_download("rhasspy/piper-checkpoints", allf[0], repo_type="dataset")   # the first one; pick another from the list if you prefer
print("Starting from", allf[0])

In [ ]:
#@title 5. Train (this is the long step: from about one hour to several hours)
!cd piper1-gpl && python3 -m piper.train fit \
  --data.voice_name "{VOICE_NAME}" \
  --data.csv_path ../data/metadata.csv \
  --data.audio_dir ../data/wavs \
  --model.sample_rate 22050 \
  --data.espeak_voice "{ESPEAK}" \
  --data.cache_dir ../cache \
  --data.config_path ../out/{VOICE_NAME}.onnx.json \
  --data.batch_size {BATCH_SIZE} \
  --ckpt_path "{BASE_CKPT}" \
  --trainer.max_epochs {EPOCHS}

In [ ]:
#@title 6. Export to ONNX
import glob, os
ckpts = sorted(glob.glob("piper1-gpl/lightning_logs/*/checkpoints/*.ckpt"), key=os.path.getmtime)
print(ckpts[-3:]); CKPT = ckpts[-1]
!cd piper1-gpl && python3 -m piper.train.export_onnx --checkpoint ../{CKPT} --output-file ../out/{VOICE_NAME}.onnx
import json; cfg = json.load(open(f"out/{VOICE_NAME}.onnx.json"))
print({k: cfg.get(k) for k in ("audio", "espeak", "phoneme_type", "num_speakers")})
assert cfg.get("phoneme_type", "espeak") == "espeak", "The app only supports eSpeak voices"

In [ ]:
#@title 7. Listen before you download
!pip -q install piper-tts
tests = {"en": ["Hello! I am your study assistant.", "Alarm set for tomorrow, six in the morning.", "You scored seventy two out of a hundred."],
         "hi": ["नमस्ते! मैं आपकी पढ़ाई की सहायक हूँ।", "कल सुबह छह बजे के लिए अलार्म सेट हो गया।", "आपको सौ में से बहत्तर अंक मिले।"]}[LANG]
from IPython.display import Audio, display
for i, t in enumerate(tests):
    !python3 -m piper -m out/{VOICE_NAME}.onnx -f out/test{i}.wav -- "{t}"
    display(Audio(f"out/test{i}.wav"))

In [ ]:
#@title 8. Download the two files, then add them in Abhyashify: Assistant settings → Offline voice pack
from google.colab import files
files.download(f"out/{VOICE_NAME}.onnx"); files.download(f"out/{VOICE_NAME}.onnx.json")